# 🎙️ Voicebox: Qwen3-TTS Engine.
This uses [voicebox.sh](https://voicebox.sh) and the Qwen3 TTS engine. to generate Text to voice, then save the output to your Google Drive

This does REQUIRE you to have an [exported voice profile](https://docs.voicebox.sh/overview/quick-start) from an existing install of voicebox.sh.

However that feature of Voicebox requires almost no compute power, and can be done on any hardware.

It is DISCOURAGED to run any TTS longer then 4,000 /10,000 characters for high/low quality as the notebook can crash. At the bottom is an audio merger so that you can generate your audio in parts and you can stitch them together after.

In [ ]:
# @title 🎙️ Voicebox: Qwen3-TTS Direct Generator

# @markdown ### ⚙️ **Model & Drive Settings** <br />
# @markdown Higher quality outputs take about 3X longer to load. Quality difference is marginal.
higher_quality_outputs = False # @param {type:"boolean"}
# @markdown This saves the output in your Google Drive under /Voicebox/Project_name/  as you have entered it here:
project_name = "Project_name" # @param ["Project_name"] {allow-input: true}

# @markdown ---
# @markdown ### 📝 **Text Input**
text_to_speak = "Oh! Hello there! Wait, can you actually hear me?! Yay, I have a voice now!" # @param ["Oh! Hello there! Wait, can you actually hear me?! Yay, I have a voice now!"] {allow-input: true}

# @markdown ---
# @markdown ### 📁 **Voice Profile** <br />
# @markdown This does REQUIRE you to have an [exported voice profile](https://docs.voicebox.sh/overview/quick-start) from an existing install of voicebox.sh It prompts you to upload it in runtime. <br />
# @markdown Your voice profile must contain a transcript of the example, if it does not, you can manually add that here.
manual_ref_text = "" # @param {type:"string"}

import sys
import os
import shutil
import zipfile
import json
import time
import re
import glob

# 1. System Dependencies & Repository Setup
print("⏳ Step 1/5: Installing system packages and cloning Voicebox repository...")
!apt-get install -y -qq sox libsox-fmt-all > /dev/null 2>&1
!rm -rf /content/voicebox
!git clone https://github.com/jamiepine/voicebox.git /content/voicebox

# 2. Python Package Installation
print("📦 Step 2/5: Installing core Python dependencies...")
!pip install -q -U qwen-tts accelerate soundfile

# 3. Imports & Logging Configuration
import torch
import numpy as np
import soundfile as sf
import transformers
import warnings

transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

from IPython.display import Audio, display, clear_output
from google.colab import drive, files
from qwen_tts import Qwen3TTSModel

print("✅ Setup complete! Preparing Google Drive & Voice Profile...")

# 4. Mount Google Drive
from IPython.display import clear_output
clear_output()
print("\n📂 Step 3/5: Mounting Google Drive...")
drive.mount('/content/drive')
drive_output = f'/content/drive/MyDrive/Voicebox/{project_name}/'
os.makedirs(drive_output, exist_ok=True)
print(f"✅ Output folder ready: {drive_output}")

# 5. Interactive Upload Logic
clear_output(wait=True)
print("⬆️ Please upload your Voicebox profile (.zip):")
uploaded = files.upload()
if uploaded:
    uploaded_filename = list(uploaded.keys())[0]
    profile_zip_path = os.path.join('/content', uploaded_filename)
    print(f"✅ Uploaded successfully: {profile_zip_path}")
else:
    raise FileNotFoundError("❌ No .zip file provided.")

# 6. Load Qwen3-TTS Model
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-Base" if higher_quality_outputs else "Qwen/Qwen3-TTS-12Hz-0.6B-Base"
model_label = "1.7B" if higher_quality_outputs else "0.6B"

print(f"\n🧠 Step 5/5: Loading Qwen3-TTS {model_label} Model into GPU VRAM...")

# Choose bfloat16 if supported, otherwise float32 to prevent CUDA assertion faults
target_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32

model = Qwen3TTSModel.from_pretrained(
    model_id,
    torch_dtype=target_dtype,
    device_map="auto"
)

print(f"✅ {model_label} Model loaded into GPU VRAM successfully!")

# 7. Text Chunking Helper
def chunk_text(text, max_chars=300):
    sentences = re.split(r'(?<=[.!?\n])\s+', text)
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chars:
            current_chunk += sentence + " "
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

# 8. Unpack Voice Profile & Run TTS Generation
print("\n🎙️ Unpacking Voice Profile & Generating Speech...")
extract_path = "/content/temp_voice_profile"
os.makedirs(extract_path, exist_ok=True)

try:
    with zipfile.ZipFile(profile_zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    ref_audio = None
    ref_text = manual_ref_text.strip() if manual_ref_text else ""

    # Locate .wav audio
    for root, dirs, f_list in os.walk(extract_path):
        for file in f_list:
            if file.endswith(".wav") and not ref_audio:
                ref_audio = os.path.join(root, file)

    # Locate transcript in samples.json
    if not ref_text and ref_audio:
        audio_basename = os.path.basename(ref_audio)
        for root, dirs, f_list in os.walk(extract_path):
            if "samples.json" in f_list:
                try:
                    with open(os.path.join(root, "samples.json"), 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        if isinstance(data, list):
                            for item in data:
                                if isinstance(item, dict) and "text" in item:
                                    if "audio" in item and audio_basename in item["audio"]:
                                        ref_text = item["text"]
                                        break
                                    if not ref_text:
                                        ref_text = item["text"]
                except Exception as e:
                    print(f"⚠️ Could not parse samples.json: {e}")

    if not ref_audio:
        raise FileNotFoundError("❌ No reference .wav audio found inside the .zip profile.")

    output_filename = os.path.join(drive_output, f"voicebox_output_{int(time.time())}.wav")
    text_chunks = chunk_text(text_to_speak, max_chars=400)
    all_audio_arrays = []
    final_sample_rate = 24000

    print(f"⚡ Generating speech across {len(text_chunks)} chunk(s)...")

    for i, chunk in enumerate(text_chunks, 1):
        if not chunk.strip():
            continue
        print(f"   └─ Processing chunk {i}/{len(text_chunks)}...")
        if ref_text:
            wavs, sr = model.generate_voice_clone(
                text=chunk,
                language="English",
                ref_audio=ref_audio,
                ref_text=ref_text
            )
        else:
            wavs, sr = model.generate_voice_clone(
                text=chunk,
                language="English",
                ref_audio=ref_audio,
                x_vector_only_mode=True
            )
        all_audio_arrays.append(wavs[0])
        final_sample_rate = sr

    if all_audio_arrays:
        final_audio = np.concatenate(all_audio_arrays)
        sf.write(output_filename, final_audio, final_sample_rate)
        print(f"\n🎉 Success! Audio saved to: {output_filename}\n")
        display(Audio(output_filename))
    else:
        print("❌ Generation failed. Check your input text.")

finally:
    if os.path.exists(extract_path):
        shutil.rmtree(extract_path)

In [ ]:
# @title 🔗 Voicebox: Merge Project Audio Files.

# @markdown ### 📁 **Select Project**
# @markdown Enter the project name to merge multiple audio files. Leaving this blank defaults to the last run project from above. <br />
# @markdown This merges the files in alphabetical order of name, so will work fine aslong as the above is generated in the desired order.
project_name = "" # @param {type:"string"}
merged_output_filename = "merged_voicebox_output.wav" # @param {type:"string"}

import os
import glob
import re
import numpy as np
import soundfile as sf
from google.colab import drive

# 1. Mount Google Drive if not already mounted
if not os.path.exists('/content/drive/MyDrive'):
    print("📂 Mounting Google Drive...")
    drive.mount('/content/drive')

# 2. Resolve Project Directory
target_project = project_name.strip() if project_name.strip() else "Project_name"
project_dir = f'/content/drive/MyDrive/Voicebox/{target_project}/'

print(f"🔍 Searching for audio files in: {project_dir}")

if not os.path.exists(project_dir):
    raise FileNotFoundError(f"❌ Project directory does not exist: {project_dir}")

# 3. Locate and Sort Output .wav Files
wav_files = [
    f for f in glob.glob(os.path.join(project_dir, "*.wav"))
    if not os.path.basename(f).startswith("merged_")
]

if not wav_files:
    raise FileNotFoundError(f"❌ No output .wav files found in {project_dir} to merge.")

# Sort alphabetically using natural sorting (so "part2" comes before "part10")
wav_files.sort(key=lambda x: [int(c) if c.isdigit() else c.lower() for c in re.split(r'(\d+)', os.path.basename(x))])

print(f"Found {len(wav_files)} file(s) to merge in alphabetical order:")
for i, f in enumerate(wav_files, 1):
    print(f"   {i}. {os.path.basename(f)}")

# 4. Read and Concatenate Audio
audio_segments = []
target_sr = None

for file_path in wav_files:
    data, sr = sf.read(file_path)
    if target_sr is None:
        target_sr = sr
    elif sr != target_sr:
        print(f"⚠️ Warning: Sample rate mismatch on {os.path.basename(file_path)}. Resampling...")
        import librosa
        data = librosa.resample(data, orig_sr=sr, target_sr=target_sr)

    audio_segments.append(data)

# Combine into a single NumPy array
merged_audio = np.concatenate(audio_segments)

# 5. Save and Display Merged Output
save_path = os.path.join(project_dir, merged_output_filename)
sf.write(save_path, merged_audio, target_sr)

print(f"\n🎉 Successfully merged {len(wav_files)} files!")
print(f"💾 Saved to: {save_path}\n")

from IPython.display import Audio, display
display(Audio(save_path))